# Nettoyage de la source FranceConnect — vers le schéma PSP

Deuxième des 4 étapes décrites dans [README.md](README.md).

Transforme l'export brut de `eligibility_results` laissé par `export_eligible_pending.sql`
en un CSV au schéma de la base de production, prêt pour `../generate_new_codes.ipynb`.

Cette source n'est pas un fichier partenaire : ce sont les personnes que le site a jugées
éligibles sans que LCA ait pu leur servir un code (verdict `eligible_pending`). Trois choses
manquent donc à l'export et sont reconstruites ici, dans `clean_fc_lib.py` :

- la **situation** — `eligibility_results` ne mémorise pas quelle aide a ouvert le droit,
  elle se redéduit des réponses brutes d'API Particulier ;
- le **genre des enfants** — absent de `enfant_identite`, retrouvé dans le tableau `enfants`
  de la réponse quotient familial ;
- le **schéma PSP** — l'identité arrive au vocabulaire FranceConnect, répartie sur deux
  colonnes JSON selon `source`.

L'enchaînement lui-même vit dans [fc_pipeline.py](fc_pipeline.py), que ce notebook et la cron
[run_fc_pipeline.sh](run_fc_pipeline.sh) appellent tous les deux : passer à la main ou passer
automatiquement fait donc exactement la même chose. En ligne de commande, cette étape s'écrit
`python fc_pipeline.py clean`.

## Prérequis
`export_eligible_pending.sql` a tourné et écrit `FC_EXPORT_PATHFILE_2026`.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# fc_pipeline.py vit à côté de ce notebook et rend lui-même importables partners_lib et
# utils.data_utils ; ce chemin-ci n'est là que pour le trouver, lui.
notebook_dir = str(Path.cwd())
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

import fc_pipeline as pipeline

load_dotenv()

input_filepath = os.environ['FC_EXPORT_PATHFILE_2026']
output_filepath = os.environ['DB_FC_EXPORT_2026']

print(f"entrée : {input_filepath}")
print(f"sortie : {output_filepath}")

In [ ]:
# Tout l'enchaînement, dans fc_pipeline.clean : appariement du genre des enfants, projection
# vers le schéma PSP, situation et organisme, sérialisation des deux colonnes JSON, filtre
# des lignes incomplètes, déduplication, écriture. L'ordre des étapes est contraint, chacune
# consommant la précédente — d'où une fonction plutôt qu'une suite de cellules : la cron
# (run_fc_pipeline.sh) appelle exactement la même.
stats = pipeline.clean(input_filepath, output_filepath)

pipeline.print_stats(stats)

## ⏭ Étape suivante : `../generate_new_codes.ipynb` avec `SOURCE = 'FC'`

Puis, **et seulement une fois le CSV final produit**, `writeback_codes.ipynb` et
`writeback_verdict.sql` pour marquer ces bénéficiaires en base. Sans ce dernier passage, le
prochain export les reprendrait et leur fabriquerait un second code.